# LeetCode #72: Edit Distance

https://leetcode.com/problems/edit-distance/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(3^{\max(m,n)})$ | $O(m+n)$ |
| **Optimal: 1D DP ★** | $O(m \times n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
Recursively try all three operations (insert, delete, replace) at each mismatch, branching exponentially. The same sub-problems are recomputed many times without memoization.

### Optimal: 1D DP ★
Build the edit-distance table row by row using a single 1D array. `dp[j]` = minimum edits to convert `word1[0..i-1]` to `word2[0..j-1]`. Track the diagonal predecessor in a variable (`prev`) to avoid overwriting it before it is used. This shrinks space from $O(mn)$ to $O(n)$.

**Constraints:**
* $0 \leq \text{word1.length}, \text{word2.length} \leq 500$


## Solutions

### C#

In [ ]:
public class Solution {
    public int MinDistance(string word1, string word2) {
        int m = word1.Length, n = word2.Length;

        // dp[j] = edit distance between word1[0..i-1] and word2[0..j-1]
        int[] dp = new int[n + 1];

        // Base: converting empty word1 prefix to word2 prefix = all insertions
        for (int j = 0; j <= n; j++) dp[j] = j;

        for (int i = 1; i <= m; i++) {
            int prev = dp[0]; // Holds dp[i-1][j-1] before overwriting
            dp[0] = i;        // Converting word1[0..i-1] to empty = i deletions

            for (int j = 1; j <= n; j++) {
                int temp = dp[j];
                if (word1[i - 1] == word2[j - 1]) {
                    // Characters match: no extra operation needed
                    dp[j] = prev;
                } else {
                    // Take the cheapest of: replace (prev), delete (dp[j]), insert (dp[j-1])
                    dp[j] = Math.Min(prev, Math.Min(dp[j], dp[j - 1])) + 1;
                }
                prev = temp;
            }
        }

        return dp[n];
    }
}

### Python

In [ ]:
class Solution:
    def minDistance(self, word1: str, word2: str) -> int:
        m, n = len(word1), len(word2)

        # dp[j] = edit distance between word1[0..i-1] and word2[0..j-1]
        dp = list(range(n + 1))  # Base: all insertions for empty word1

        for i in range(1, m + 1):
            prev = dp[0]  # Holds dp[i-1][j-1] before overwriting
            dp[0] = i     # Converting word1[0..i-1] to empty = i deletions

            for j in range(1, n + 1):
                temp = dp[j]
                if word1[i - 1] == word2[j - 1]:
                    # Characters match: no extra operation needed
                    dp[j] = prev
                else:
                    # Take the cheapest of: replace (prev), delete (dp[j]), insert (dp[j-1])
                    dp[j] = min(prev, dp[j], dp[j - 1]) + 1
                prev = temp

        return dp[n]


### Go

In [ ]:
func minDistance(word1 string, word2 string) int {
    m, n := len(word1), len(word2)

    // dp[j] = edit distance between word1[0..i-1] and word2[0..j-1]
    dp := make([]int, n+1)
    for j := 0; j <= n; j++ {
        dp[j] = j // Base: all insertions for empty word1
    }

    for i := 1; i <= m; i++ {
        prev := dp[0] // Holds dp[i-1][j-1] before overwriting
        dp[0] = i     // Converting word1[0..i-1] to empty = i deletions

        for j := 1; j <= n; j++ {
            temp := dp[j]
            if word1[i-1] == word2[j-1] {
                // Characters match: no extra operation needed
                dp[j] = prev
            } else {
                // Take the cheapest of: replace (prev), delete (dp[j]), insert (dp[j-1])
                minVal := prev
                if dp[j] < minVal {
                    minVal = dp[j]
                }
                if dp[j-1] < minVal {
                    minVal = dp[j-1]
                }
                dp[j] = minVal + 1
            }
            prev = temp
        }
    }

    return dp[n]
}

### Rust

In [ ]:
impl Solution {
    pub fn min_distance(word1: String, word2: String) -> i32 {
        let w1 = word1.as_bytes();
        let w2 = word2.as_bytes();
        let (m, n) = (w1.len(), w2.len());

        // dp[j] = edit distance between word1[0..i-1] and word2[0..j-1]
        let mut dp: Vec<i32> = (0..=(n as i32)).collect(); // Base: all insertions

        for i in 1..=m {
            let mut prev = dp[0]; // Holds dp[i-1][j-1] before overwriting
            dp[0] = i as i32;    // Converting word1[0..i-1] to empty = i deletions

            for j in 1..=n {
                let temp = dp[j];
                dp[j] = if w1[i - 1] == w2[j - 1] {
                    // Characters match: no extra operation needed
                    prev
                } else {
                    // Take the cheapest of: replace (prev), delete (dp[j]), insert (dp[j-1])
                    prev.min(dp[j]).min(dp[j - 1]) + 1
                };
                prev = temp;
            }
        }

        dp[n]
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `word1 = "horse", word2 = "ros"`
Three operations: replace `h` with `r`, delete `r`, delete `e` — edit distance 3. The DP table fills in row-by-row, reaching `dp[3] = 3` at the end.

### 2. Slightly Complex
**Input:** `word1 = "intention", word2 = "execution"`
Edit distance 5 (replace `i` with `e`, replace `n` with `x`, replace `t` with `c`, insert `u`, replace `n` with `n`... several combos). The 9-by-9 table resolves this in $O(81)$ steps.

### 3. Edge Case: Time Factor
**Input:** `word1 = word2 = "abcde..."` (500 characters each, completely different)
The DP fills $500 \times 500 = 250{,}000$ cells — the worst-case time path — and every cell triggers the three-way minimum.

### 4. Edge Case: Space Factor
**Input:** `word1 = ""`, `word2 = "abc"`
Empty word1: the answer is `len(word2) = 3` (three insertions). The DP array has length 4, and `dp[0] = 0` stays fixed — $O(n)$ is minimised here since `m = 0`.

### 5. Almost-Impossible but Plausible
**Input:** `word1 = "ab"`, `word2 = "ba"`
Intuitively a single swap, but edit distance only supports insert/delete/replace — not swap. The true distance is 2 (replace `a` with `b` and `b` with `a`). The DP correctly computes 2, not 1, exposing the no-swap constraint.
